# Hybrid Ransomware EDR — Demo Presentation Notebook

**Project:** Ransomware Detection and Mitigation System  
**Module:** Computer and Network Security  
**Purpose:** This short notebook is designed for the **video demo / presentation**.  

It does **not** retrain the full pipeline. The full research notebook remains:

```text
notebooks/03_Research_Grade_Pipeline.ipynb
```

This notebook only presents the final system architecture, final results, decision policy, and demo commands.

## 1 — Presentation Flow

Use this notebook for the explanation part of the video, then switch to the terminal demo script.

Recommended flow:

1. Explain the ransomware problem.
2. Explain the hybrid EDR architecture.
3. Explain Layer 1: static memory forensics.
4. Explain Layer 2: dynamic behavioral telemetry.
5. Explain the orchestrator decision policy.
6. Show final evaluation results.
7. Run terminal demo scenarios:
   - Notepad benign
   - Hard benign workload
   - Dynamic ransomware simulation
   - Real memory ransomware row / Layer 1 critical case
8. End with limitations and future work.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)

def show_table(title, rows):
    display(Markdown(f"### {title}"))
    display(pd.DataFrame(rows))

display(Markdown("Notebook loaded. This is a presentation/demo notebook, not a full training notebook."))

## 2 — System Architecture

The system is a **Hybrid Endpoint Detection and Response (EDR)** prototype.

It combines:

- **Layer 1:** Static memory forensics
- **Layer 2:** Dynamic behavioral telemetry
- **Orchestrator:** Risk fusion and mitigation decision
- **Mitigation engine:** log, alert, suspend, kill, quarantine, block

```text
                  ┌────────────────────────────┐
                  │      Monitored Process      │
                  └──────────────┬─────────────┘
                                 │
             ┌───────────────────┴───────────────────┐
             │                                       │
             ▼                                       ▼
┌────────────────────────────┐       ┌────────────────────────────┐
│ Layer 1: Static Memory      │       │ Layer 2: Dynamic Telemetry  │
│ Forensics                   │       │ CNN Behavioral Monitoring   │
│                             │       │                             │
│ RF / XGBoost / LightGBM     │       │ CPU, reads, writes, files   │
│ Autoencoder anomaly support │       │ CNN over 8-tick windows     │
└──────────────┬─────────────┘       └──────────────┬─────────────┘
               │                                    │
               └─────────────────┬──────────────────┘
                                 ▼
                 ┌──────────────────────────────┐
                 │ Hybrid EDR Orchestrator       │
                 │ Risk state + evidence gate    │
                 │ Risk accumulator + policy     │
                 └──────────────┬───────────────┘
                                ▼
                 ┌──────────────────────────────┐
                 │ Mitigation Decision           │
                 │ log / alert / suspend / kill  │
                 └──────────────────────────────┘
```

## 3 — Layer 1: Static Memory Forensics

Layer 1 analyzes memory-forensics features extracted from memory snapshots.

It answers:

> **Does this process/system memory look forensically malicious?**

Important feature groups:

- `malfind.*` → code injection indicators
- `psxview.*` → hidden process inconsistencies
- `ldrmodules.*` → suspicious module/DLL anomalies
- `handles.*` → abnormal file, mutex, event, registry, and thread handles
- `pslist.*` → process hierarchy and thread/process statistics
- `svcscan.*` → service and driver manipulation

The supervised Layer 1 detector is a **RF / XGBoost / LightGBM ensemble**.  
The Autoencoder is used only as an **anomaly / novelty support signal**, not as the main ransomware classifier.

In [ ]:
layer1_results = [
    {
        "Component": "RF / XGBoost / LightGBM ensemble",
        "Role": "Primary ransomware memory classifier",
        "Final use": "Produces Layer 1 memory risk probability/state"
    },
    {
        "Component": "Autoencoder",
        "Role": "Anomaly / novelty support",
        "Final use": "Boosts WATCH/SUSPICIOUS state when memory is unusual"
    },
    {
        "Component": "Layer 1 state",
        "Role": "Risk prior for orchestrator",
        "Final use": "SAFE / WATCH / SUSPICIOUS / HIGH_RISK / CRITICAL"
    },
]
show_table("Layer 1 Components", layer1_results)

layer1_metric_rows = [
    {"Layer 1 model": "Default V8 ensemble", "Recall": "≈ 91.78%", "Use": "Main balanced Layer 1 detector"},
    {"Layer 1 model": "High-recall V8 ensemble", "Recall": "≈ 94.84%", "Use": "Aggressive / high-sensitivity mode"},
    {"Layer 1 model": "Autoencoder", "Recall": "High but low precision", "Use": "Anomaly support, not final classifier"},
]
show_table("Layer 1 Final Results Summary", layer1_metric_rows)

## 4 — Layer 2: Dynamic Behavioral Telemetry

Layer 2 monitors runtime behavior continuously.

It answers:

> **Is the process currently behaving like ransomware encryption?**

The CNN receives telemetry windows with 8 ticks and 10 features.

Base features:

- `cpu_percent`
- `memory_rss_mb`
- `memory_vms_mb`
- `io_read_bytes_delta`
- `io_write_bytes_delta`
- `net_bytes_sent_delta`
- `num_open_files`

Derived ransomware-behavior features:

\[
write\_read\_ratio =
\frac{io\_write\_bytes\_delta}{io\_read\_bytes\_delta + 1}
\]

\[
cpu\_x\_write =
cpu\_percent \times io\_write\_bytes\_delta
\]

\[
io\_write\_intensity =
\frac{io\_write\_bytes\_delta}{memory\_rss\_mb \times 1024 + 1}
\]

Layer 2 is useful for detecting the **read → encrypt → write** pattern, but it is not trusted alone for hard mitigation.

In [ ]:
layer2_metrics = [
    {
        "Metric": "Recall",
        "Value": "57.42%",
        "Interpretation": "CNN catches part of active encryption windows"
    },
    {
        "Metric": "Precision",
        "Value": "29.98%",
        "Interpretation": "CNN alone is not enough for final mitigation"
    },
    {
        "Metric": "False Positive Rate",
        "Value": "11.12%",
        "Interpretation": "Much cleaner than early versions; still filtered by orchestrator"
    },
    {
        "Metric": "Specificity",
        "Value": "88.88%",
        "Interpretation": "Most benign windows are rejected"
    },
    {
        "Metric": "Calibration",
        "Value": "Platt",
        "Interpretation": "CNN scores are calibrated before thresholding"
    },
]
show_table("Layer 2 CNN Window-Level Metrics", layer2_metrics)

## 5 — Orchestrator Decision Policy

The orchestrator is the main EDR decision engine.

It combines:

- Layer 1 memory risk state
- Layer 2 CNN probability
- encryption-evidence gate
- ransomware-specific evidence score
- evidence streak / risk accumulator
- mitigation safety rules

The final policy is:

```text
SAFE/WATCH + no encryption evidence
→ log only

SAFE/WATCH + generic high I/O
→ ALERT_ONLY

SAFE/WATCH + ransomware-specific sustained encryption behavior
→ SOFT_BLOCK

SUSPICIOUS/HIGH_RISK + ransomware-specific sustained evidence
→ HARD_BLOCK allowed

CRITICAL Layer 1 memory evidence
→ HARD response allowed
```

Important rule:

> **Layer 2 cannot hard-kill when Layer 1 is only SAFE or WATCH.**

In [ ]:
policy_rows = [
    {
        "Layer 1 state": "SAFE",
        "Layer 2 behavior": "low/no evidence",
        "Decision": "log_event",
        "Reason": "No meaningful threat evidence"
    },
    {
        "Layer 1 state": "SAFE/WATCH",
        "Layer 2 behavior": "generic high I/O",
        "Decision": "ALERT_ONLY",
        "Reason": "Avoid suspending backup/copy/compression workloads"
    },
    {
        "Layer 1 state": "SAFE/WATCH",
        "Layer 2 behavior": "ransomware-specific sustained encryption behavior",
        "Decision": "SOFT_BLOCK",
        "Reason": "Contain possible encryption while avoiding hard kill"
    },
    {
        "Layer 1 state": "SUSPICIOUS/HIGH_RISK",
        "Layer 2 behavior": "ransomware-specific sustained evidence",
        "Decision": "HARD_BLOCK allowed",
        "Reason": "Memory and behavior agree"
    },
    {
        "Layer 1 state": "CRITICAL",
        "Layer 2 behavior": "not required",
        "Decision": "HARD response allowed",
        "Reason": "Strong memory-forensic ransomware evidence"
    },
]
show_table("Final EDR Decision Policy", policy_rows)

## 6 — Final Regression Outcomes

These are the final expected system-level results used for the presentation.

The important point is that the system acts safely:

- It does **not** block Notepad.
- It does **not** suspend hard benign workloads.
- It **soft-blocks** dynamic ransomware behavior.
- It allows hard response only when Layer 1 reaches critical memory risk.

In [ ]:
regression_rows = [
    {
        "Scenario": "Notepad benign",
        "Expected behavior": "No alert / no block",
        "Final action": "log_event",
        "Status": "PASS"
    },
    {
        "Scenario": "Hard benign workloads",
        "Expected behavior": "No suspend / no hard block",
        "Final action": "ALERT_ONLY: send_alert + log_event",
        "Status": "PASS"
    },
    {
        "Scenario": "Dynamic ransomware simulation",
        "Expected behavior": "Detect active encryption and contain",
        "Final action": "SOFT_BLOCK: suspend_process + send_alert + log_event",
        "Status": "PASS"
    },
    {
        "Scenario": "Real ransomware memory row",
        "Expected behavior": "Layer 1 CRITICAL memory verdict",
        "Final action": "HARD response allowed",
        "Status": "PASS"
    },
    {
        "Scenario": "Layer 2 hard-kill safety",
        "Expected behavior": "No kill when Layer 1 is SAFE/WATCH",
        "Final action": "Hard-kill blocked by policy",
        "Status": "PASS"
    },
]
show_table("Final Process-Level Regression Tests", regression_rows)

hard_benign_summary = [
    {
        "Files tested": 11,
        "Alert-only count": 11,
        "Soft-block count": 0,
        "Hard-block count": 0,
        "Interpretation": "Hard benign high-I/O workloads are not suspended or killed"
    }
]
show_table("Hard Benign Negative Control Summary", hard_benign_summary)

## 7 — Terminal Demo Commands

Use the terminal script for the **live demo part** of the video.

The notebook is only for explanation and final results. The terminal script is what shows the EDR running like a real monitor.

Important: this demo section does **not** use training CSV replay. The tests are live or generated inside the demo:

```bash
# 1. Open Notepad manually, then monitor it live
python demo_realtime_edr.py --mode live --name notepad.exe --duration 30

# Linux alternative if you are not using Windows
python demo_realtime_edr.py --mode live --name gedit --duration 30

# 2. Generated hard-benign workload
# This simulates backup/copy/compression-like activity on dummy files.
python demo_realtime_edr.py --scenario hard_benign_dummy --duration 35

# 3. Generated ransomware-like dummy workload
# This is NOT real ransomware. It only works on dummy files.
python demo_realtime_edr.py --scenario ransomware_dummy --duration 35

# 4. Optional controlled mitigation on the dummy ransomware process only
python demo_realtime_edr.py --scenario ransomware_dummy --duration 35 --allow-demo-mitigation

# 5. Layer 1 static memory-forensics demonstration
python demo_realtime_edr.py --scenario memory_critical
```

For safety, keep dry-run mode by default. In the video, explain:

> “The demo does not run real ransomware and does not replay training data. It uses live process monitoring and generated dummy workloads inside a controlled VM.”



In [ ]:
demo_commands = [
    {
        "Demo": "Live Notepad benign",
        "Command": "python demo_realtime_edr.py --mode live --name notepad.exe --duration 30",
        "Expected": "SAFE / log_event / no alert / no block"
    },
    {
        "Demo": "Generated hard benign workload",
        "Command": "python demo_realtime_edr.py --scenario hard_benign_dummy --duration 35",
        "Expected": "SAFE or ALERT_ONLY; no suspend; no kill"
    },
    {
        "Demo": "Generated ransomware dummy",
        "Command": "python demo_realtime_edr.py --scenario ransomware_dummy --duration 35",
        "Expected": "SOFT_BLOCK / suspend + alert + log, or dry-run WOULD SUSPEND"
    },
    {
        "Demo": "Controlled mitigation demo",
        "Command": "python demo_realtime_edr.py --scenario ransomware_dummy --duration 35 --allow-demo-mitigation",
        "Expected": "Mitigation allowed only on the dummy child process"
    },
    {
        "Demo": "Layer 1 memory critical case",
        "Command": "python demo_realtime_edr.py --scenario memory_critical",
        "Expected": "Layer 1 CRITICAL / hard response allowed"
    },
]
show_table("Terminal Demo Command Plan", demo_commands)



## 8 — Demo Safety Notes

For the video demo:

- Use a VM if possible.
- Do not run real ransomware.
- Do not use training CSV replay for the live demo.
- Use only safe dummy workloads generated by the demo script.
- Keep mitigation in dry-run mode by default.
- Only use `--allow-demo-mitigation` for the controlled dummy ransomware process.
- Do not allow the script to kill or suspend system processes.
- Keep the demo workspace limited to dummy files only.
- Explain that Layer 1 live memory extraction is heavier than Layer 2 telemetry, so the demo uses prepared memory-feature rows for the memory-forensics case.

Recommended sentence:

> “Layer 2 runs live because telemetry is lightweight. Layer 1 is a static memory-forensics layer, so in the demo we show it using prepared memory-feature rows. In a full EDR deployment, Layer 1 would run periodically or when Layer 2 raises suspicion.”



## 9 — Limitations and Future Work

This project should be presented honestly as a **research-grade hybrid EDR prototype**, not as a commercial production antivirus.

Current limitations:

- Layer 2 CNN is useful but not trusted alone for hard mitigation.
- Layer 1 live extraction is represented through prepared memory features, not full live Volatility acquisition.
- Hard benign workloads are difficult because backup/compression/copy operations can resemble ransomware at the I/O level.
- Explainability methods such as SHAP and counterfactual tests are diagnostic and need further validation.
- More real-world benign and ransomware-like workloads should be tested.

Future work:

- Add more hard benign workloads.
- Improve CNN probability calibration.
- Add real-time memory feature extraction.
- Improve process-level time-to-detection metrics.
- Add a richer analyst dashboard.
- Test against more ransomware families and unseen behaviors.

## 10 — Closing Message

Final presentation message:

> “The system is a hybrid ransomware EDR prototype. It combines static memory forensics, dynamic telemetry, and a risk-aware orchestrator. The main contribution is not one model, but the safe fusion of memory evidence and behavioral evidence to reduce false positives while still responding to ransomware-like activity.”